# RFDE Rigorous Evaluation: Demo Notebook

## Overview

This notebook demonstrates the **Rigorous Evaluation of RFDE** (Demand-Driven Fact Extraction) vs. baseline methods (CoT, RAG, LINC) on 26 logical reasoning tasks.

**What this evaluation measures:**
1. **Independent Predicate Verification**: Hallucination rate via document-span regex matching
2. **Per-Class Accuracy**: Yes/no breakdown by method
3. **Atomic Extraction Precision/Recall**: RFDE leaf-predicate accuracy
4. **McNemar's Paired Tests**: Statistical significance vs. baselines
5. **Expected Calibration Error (ECE)**: Confidence calibration
6. **Decomposition Type Stratification**: Accuracy per task complexity
7. **Inference Latency**: RFDE performance metrics
8. **Protocol Violation Flags**: Data quality warnings

**Key Finding**: RFDE achieves 100% answer accuracy but 12.5% hallucination rate when measuring at the predicate level.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
from loguru import logger
import json
import sys
import re
import math
import gc
from collections import defaultdict, Counter
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binomtest

# Configure logger
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-57cbed-demand-driven-fact-extraction-for-neuro/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub or local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    # Fallback: local file
    import os
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()
logger.info(f"Loaded data with {len(data['datasets'][0]['examples'])} examples and {len(data['datasets'][1]['examples'])} summary rows")
print(f"\nDataset metadata:")
print(f"  Evaluation: {data['metadata']['evaluation_name']}")
print(f"  Methods: {', '.join(data['metadata']['methods'])}")
print(f"  Class distribution: {data['metadata']['class_distribution']}")

In [ ]:
# Config: All tunable parameters. Start with MINIMAL values for quick demo.
# Increase step-by-step during testing if time permits.

METHODS = ["rfde", "cot", "rag", "linc"]
N_BINS_ECE = 10  # Bins for calibration analysis
MAX_EXAMPLES_DISPLAY = 3  # Examples to show in results

## Helper Functions

Extract and analyze proof traces from examples to classify task types and measure hallucination.

In [ ]:
def parse_proof_trace(example: dict) -> dict | None:
    raw = example.get("metadata_proof_trace", "")
    if not raw:
        return None
    try:
        return json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        return None

def classify_decomposition_type(trace: dict | None, ground_truth: str) -> str:
    if trace is None:
        return "unknown"
    root_src = trace.get("source", "unknown")
    if root_src in ("root", "llm", "success"):
        return "atomic_only"
    if root_src == "rule":
        children = trace.get("children", [])
        if any(c.get("source") == "rule" for c in children):
            return "mixed"
        return "derived_required"
    return "unknown"

def count_leaf_predicates_in_trace(trace: dict | None) -> int:
    if trace is None:
        return 0
    def _count(node: dict) -> int:
        src = node.get("source", "")
        children = node.get("children", [])
        if not children:
            return 1 if src in ("llm", "success", "root") else 0
        return sum(_count(c) for c in children)
    return _count(trace)

## Metric 1: Independent Predicate Verification

Compute hallucination rate: percentage of positive predictions not supported by the document.

In [ ]:
def extract_document_from_input(input_str: str) -> str:
    m = re.search(r"Document:\s*(.+?)\s*\|\s*Query:", input_str, re.DOTALL)
    if m:
        return m.group(1).strip()
    return input_str

def verify_predicate_in_document(predicate_goal: str, document: str, asserted_answer: str) -> bool:
    if asserted_answer != "yes":
        return True
    doc_lower = document.lower()
    m = re.match(r"(\w+)\(([^,]+),\s*([^)]+)\)", predicate_goal)
    if m:
        relation, ent1, ent2 = m.group(1).lower(), m.group(2).lower().strip(), m.group(3).lower().strip()
        ent1_present = ent1 in doc_lower
        ent2_present = ent2 in doc_lower
        if ent1_present and ent2_present:
            return True
        return False
    m_unary = re.match(r"(\w+)\(([^)]+)\)", predicate_goal)
    if m_unary:
        relation, ent = m_unary.group(1).lower(), m_unary.group(2).lower().strip()
        return ent in doc_lower or (ent in doc_lower and relation in doc_lower)
    words = re.findall(r"\w+", predicate_goal.lower())
    return sum(w in doc_lower for w in words) >= len(words) * 0.6

def compute_hallucination_rate_independent(examples: list[dict], method: str) -> dict:
    halluc_count = 0
    total_assertions = 0
    for ex in examples:
        gt = ex["output"].strip().lower()
        pred = ex.get(f"predict_{method}", "").strip().lower()
        if pred == "yes":
            total_assertions += 1
            if gt == "no":
                halluc_count += 1
            elif method == "rfde":
                trace = parse_proof_trace(ex)
                if trace is not None:
                    doc = extract_document_from_input(ex["input"])
                    goal = trace.get("goal", "")
                    if goal and not verify_predicate_in_document(goal, doc, pred):
                        halluc_count += 1
    rate = halluc_count / total_assertions if total_assertions > 0 else 0.0
    return {"unsupported_predicates": halluc_count, "total_asserted": total_assertions, "hallucination_rate": rate}

# Compute hallucination for all methods
examples = data["datasets"][0]["examples"]
halluc_results = {}
for method in METHODS:
    r = compute_hallucination_rate_independent(examples, method)
    halluc_results[method] = r
    logger.info(f"  {method}: halluc_rate={r['hallucination_rate']:.4f}")

## Metric 2: Per-Class Accuracy

Compute accuracy separately for 'yes' and 'no' predictions.

In [ ]:
def compute_per_class_accuracy(examples: list[dict], method: str) -> dict:
    per_class: dict[str, list[bool]] = defaultdict(list)
    for ex in examples:
        gt = ex["output"].strip().lower()
        pred = ex.get(f"predict_{method}", "").strip().lower()
        correct = pred == gt
        per_class[gt].append(correct)
    result = {}
    all_correct = []
    for label, corrects in per_class.items():
        acc = sum(corrects) / len(corrects) if corrects else 0.0
        result[f"accuracy_{label}"] = acc
        result[f"n_{label}"] = len(corrects)
        all_correct.extend(corrects)
    result["accuracy_overall"] = sum(all_correct) / len(all_correct) if all_correct else 0.0
    return result

# Compute per-class accuracy
class_acc = {}
for method in METHODS:
    r = compute_per_class_accuracy(examples, method)
    class_acc[method] = r
    logger.info(f"  {method}: overall={r['accuracy_overall']:.4f}")

## Metric 3: Atomic Extraction Precision/Recall

RFDE leaf-predicate extraction accuracy.

In [ ]:
def compute_atomic_precision_recall(examples: list[dict]) -> dict:
    tp = fp = fn = 0
    for ex in examples:
        gt = ex["output"].strip().lower()
        pred = ex.get("predict_rfde", "").strip().lower()
        trace = parse_proof_trace(ex)
        n_leaves = max(1, count_leaf_predicates_in_trace(trace))
        if pred == gt:
            tp += n_leaves
        else:
            fp += n_leaves
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision_atomic": precision, "recall_atomic": recall, "f1_atomic": f1}

# Compute atomic metrics
atomic_pr = compute_atomic_precision_recall(examples)
logger.info(f"  RFDE: precision={atomic_pr['precision_atomic']:.4f}, recall={atomic_pr['recall_atomic']:.4f}")

## Metric 4: McNemar's Paired Significance Tests

Test if RFDE differs significantly from baselines.

In [ ]:
def mcnemar_test(labels: list[str], preds_a: list[str], preds_b: list[str]) -> dict:
    n00 = n01 = n10 = n11 = 0
    for gt, a, b in zip(labels, preds_a, preds_b):
        a_correct = (a.strip().lower() == gt.strip().lower())
        b_correct = (b.strip().lower() == gt.strip().lower())
        if a_correct and b_correct:
            n11 += 1
        elif a_correct and not b_correct:
            n10 += 1
        elif not a_correct and b_correct:
            n01 += 1
        else:
            n00 += 1
    b, c = n01, n10
    n = b + c
    if n == 0:
        return {"p_value": 1.0, "effect_size": 0.0, "significant": False, "n10": n10, "n01": n01}
    p_value = float(binomtest(c, n, 0.5).pvalue)
    effect_size = (c - b) / n if n > 0 else 0.0
    return {"p_value": p_value, "effect_size": effect_size, "significant": p_value < 0.05, "n10": n10, "n01": n01}

# Compute McNemar tests
gt_labels = [ex["output"] for ex in examples]
rfde_preds = [ex.get("predict_rfde", "") for ex in examples]
mcnemar_results = {}
for baseline in ["cot", "rag", "linc"]:
    baseline_preds = [ex.get(f"predict_{baseline}", "") for ex in examples]
    r = mcnemar_test(gt_labels, rfde_preds, baseline_preds)
    mcnemar_results[baseline] = r
    logger.info(f"  RFDE vs {baseline}: p={r['p_value']:.4f}, significant={r['significant']}")

## Metric 5: Expected Calibration Error (ECE)

Measure confidence calibration (only available for RFDE).

In [ ]:
def compute_ece(examples: list[dict], method: str, n_bins: int = 10) -> float | None:
    confs = []
    corrects = []
    for ex in examples:
        conf_str = ex.get(f"metadata_{method}_confidence")
        if conf_str is None:
            continue
        try:
            conf = float(conf_str)
        except (ValueError, TypeError):
            continue
        gt = ex["output"].strip().lower()
        pred = ex.get(f"predict_{method}", "").strip().lower()
        confs.append(conf)
        corrects.append(1.0 if pred == gt else 0.0)
    if not confs:
        return None
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confs, corrects):
        bin_idx = min(int(conf * n_bins), n_bins - 1)
        bins[bin_idx].append((conf, correct))
    ece = 0.0
    n = len(confs)
    for bin_items in bins:
        if not bin_items:
            continue
        bin_confs = [x[0] for x in bin_items]
        bin_acc = [x[1] for x in bin_items]
        mean_conf = sum(bin_confs) / len(bin_confs)
        mean_acc = sum(bin_acc) / len(bin_acc)
        ece += (len(bin_items) / n) * abs(mean_conf - mean_acc)
    return ece

# Compute ECE
ece_results = {}
for method in METHODS:
    ece = compute_ece(examples, method, N_BINS_ECE)
    ece_results[method] = ece
    logger.info(f"  {method}: ECE={ece:.4f}" if ece is not None else f"  {method}: ECE=N/A")

## Metric 6: Decomposition Type Stratification

Accuracy by task complexity (atomic vs. derived).

In [ ]:
def stratify_by_decomposition(examples: list[dict]) -> dict[str, list[dict]]:
    strata: dict[str, list[dict]] = defaultdict(list)
    for ex in examples:
        trace = parse_proof_trace(ex)
        gt = ex["output"].strip().lower()
        dtype = classify_decomposition_type(trace, gt)
        strata[dtype].append(ex)
    return dict(strata)

def compute_stratum_metrics(stratum: list[dict]) -> dict:
    if not stratum:
        return {}
    result: dict[str, Any] = {"n": len(stratum)}
    for method in METHODS:
        acc = compute_per_class_accuracy(stratum, method)
        halluc = compute_hallucination_rate_independent(stratum, method)
        result[f"{method}_accuracy"] = acc["accuracy_overall"]
        result[f"{method}_hallucination_rate"] = halluc["hallucination_rate"]
    return result

# Compute stratification
strata = stratify_by_decomposition(examples)
strata_metrics = {}
for dtype, stratum in strata.items():
    sm = compute_stratum_metrics(stratum)
    strata_metrics[dtype] = sm
    logger.info(f"  {dtype}: n={len(stratum)}, rfde_acc={sm.get('rfde_accuracy', 'N/A'):.4f}")

## Data Quality Checks

Flag protocol violations (class imbalance, small sample size).

In [ ]:
def check_protocol_violations(examples: list[dict]) -> dict:
    label_counts = Counter(ex["output"].strip().lower() for ex in examples)
    total = len(examples)
    majority_class = max(label_counts, key=lambda k: label_counts[k])
    majority_frac = label_counts[majority_class] / total if total > 0 else 0.0
    violations = []
    if majority_frac > 0.60:
        violations.append(f"imbalanced_labels: {majority_class} ({majority_frac:.1%})")
    if total < 200:
        violations.append(f"small_sample: n={total} < 200")
    return {"protocol_violation": len(violations) > 0, "violations": violations, "label_distribution": dict(label_counts), "majority_fraction": majority_frac, "n_total": total}

# Check violations
violations = check_protocol_violations(examples)
logger.info(f"Protocol violations: {violations['violations']}")

## Results Summary

Tabular and visual summary of all metrics.

In [ ]:
print("\n" + "="*80)
print("EVALUATION RESULTS SUMMARY")
print("="*80)

# Method comparison table
print("\n1. METHOD COMPARISON (Overall Accuracy & Hallucination)")
print("-" * 80)
print(f"{'Method':<10} {'Accuracy':<12} {'Halluc. Rate':<15} {'# Unsupported':<15}")
print("-" * 80)
for method in METHODS:
    acc = class_acc[method]['accuracy_overall']
    halluc = halluc_results[method]['hallucination_rate']
    unsup = halluc_results[method]['unsupported_predicates']
    print(f"{method:<10} {acc:<12.4f} {halluc:<15.4f} {unsup:<15.0f}")

# Per-class accuracy
print("\n2. PER-CLASS ACCURACY")
print("-" * 80)
print(f"{'Method':<10} {'Yes Acc':<12} {'No Acc':<12} {'N (Yes)':<10} {'N (No)':<10}")
print("-" * 80)
for method in METHODS:
    acc_yes = class_acc[method].get('accuracy_yes', -1)
    acc_no = class_acc[method].get('accuracy_no', -1)
    n_yes = class_acc[method].get('n_yes', 0)
    n_no = class_acc[method].get('n_no', 0)
    yes_str = f"{acc_yes:.4f}" if acc_yes >= 0 else "N/A"
    no_str = f"{acc_no:.4f}" if acc_no >= 0 else "N/A"
    print(f"{method:<10} {yes_str:<12} {no_str:<12} {n_yes:<10} {n_no:<10}")

# RFDE atomic metrics
print("\n3. RFDE ATOMIC EXTRACTION METRICS")
print("-" * 80)
print(f"Precision: {atomic_pr['precision_atomic']:.4f}")
print(f"Recall:    {atomic_pr['recall_atomic']:.4f}")
print(f"F1:        {atomic_pr['f1_atomic']:.4f}")

# McNemar tests
print("\n4. McNEMAR SIGNIFICANCE TESTS (RFDE vs. Baselines)")
print("-" * 80)
print(f"{'RFDE vs':<12} {'P-value':<15} {'Significant':<15} {'N10':<10} {'N01':<10}")
print("-" * 80)
for baseline, r in mcnemar_results.items():
    sig_str = "Yes" if r['significant'] else "No"
    print(f"{baseline:<12} {r['p_value']:<15.6f} {sig_str:<15} {r['n10']:<10} {r['n01']:<10}")

# Decomposition stratification
print("\n5. STRATIFICATION BY TASK COMPLEXITY")
print("-" * 80)
for dtype, sm in strata_metrics.items():
    print(f"\n{dtype.upper()} (n={sm['n']}):")
    for method in METHODS:
        acc = sm.get(f"{method}_accuracy", -1)
        halluc = sm.get(f"{method}_hallucination_rate", -1)
        acc_str = f"{acc:.4f}" if acc >= 0 else "N/A"
        halluc_str = f"{halluc:.4f}" if halluc >= 0 else "N/A"
        print(f"  {method}: acc={acc_str}, halluc={halluc_str}")

# Protocol flags
print("\n6. DATA QUALITY CHECKS")
print("-" * 80)
print(f"Total examples: {violations['n_total']}")
print(f"Class distribution: {violations['label_distribution']}")
print(f"Majority class fraction: {violations['majority_fraction']:.2%}")
if violations['violations']:
    print(f"\nProtocol violations detected:")
    for v in violations['violations']:
        print(f"  - {v}")

print("\n" + "="*80)

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RFDE Evaluation: Key Metrics', fontsize=16, fontweight='bold')

# Plot 1: Overall accuracy comparison
ax = axes[0, 0]
methods_list = list(class_acc.keys())
accs = [class_acc[m]['accuracy_overall'] for m in methods_list]
ax.bar(methods_list, accs, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_ylabel('Accuracy')
ax.set_title('Overall Accuracy by Method')
ax.set_ylim([0, 1.1])
for i, v in enumerate(accs):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Plot 2: Hallucination rate
ax = axes[0, 1]
halluc_rates = [halluc_results[m]['hallucination_rate'] for m in methods_list]
ax.bar(methods_list, halluc_rates, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_ylabel('Hallucination Rate')
ax.set_title('Hallucination Rate by Method')
ax.set_ylim([0, max(halluc_rates) * 1.3])
for i, v in enumerate(halluc_rates):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Plot 3: Stratification (RFDE only)
ax = axes[1, 0]
strata_names = list(strata_metrics.keys())
strata_accs = [strata_metrics[s].get('rfde_accuracy', 0) for s in strata_names]
ax.bar(strata_names, strata_accs, color='#1f77b4')
ax.set_ylabel('RFDE Accuracy')
ax.set_title('RFDE Accuracy by Task Type')
ax.set_ylim([0, 1.1])
for i, v in enumerate(strata_accs):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Plot 4: McNemar p-values
ax = axes[1, 1]
baselines = list(mcnemar_results.keys())
p_values = [mcnemar_results[b]['p_value'] for b in baselines]
colors_sig = ['#d62728' if p < 0.05 else '#1f77b4' for p in p_values]
ax.bar(baselines, p_values, color=colors_sig)
ax.axhline(y=0.05, color='red', linestyle='--', label='α=0.05')
ax.set_ylabel('P-value')
ax.set_title('McNemar Significance: RFDE vs Baselines')
ax.set_ylim([0, max(p_values) * 1.2])
ax.legend()
for i, v in enumerate(p_values):
    ax.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nVisualization saved to evaluation_results.png")